# 🚀 [PBL6] Huấn Luyện Tinh Chỉnh Mô Hình TPS-Inbetween (Finetune v0.1)
## 2D Anime Line Inbetweening with Arbitrary Time-Step & Memory Optimization

> **Mục tiêu:** Tinh chỉnh mô hình **TPS-Inbetween** trên tập dữ liệu anime thực tế (`anime_tps_finetune_dataset_ready`) trên **Kaggle GPU (Tesla T4 x2 hoặc P100 16GB)**.
>
> **Điểm nhấn Kỹ thuật v0.1:**
> 1. **Triệt tiêu nguy cơ CUDA OOM:** Áp dụng PyTorch AMP (FP16), Gradient Accumulation (Effective Batch Size = 8), và Random Spatial Crop $512 \times 512$ lúc huấn luyện.
> 2. **Triệt tiêu CPU RAM Leak:** Lớp `AnimeTPSLazyDataset` đọc ảnh và file `.npy` theo yêu cầu (on-the-fly), không nạp hàng loạt vào RAM 30GB của Kaggle.
> 3. **Tối ưu tốc độ suy luận:** Tận dụng 100% tọa độ điểm khớp đã tiền xử lý bằng GlueStick; loại bỏ module GlueStick khỏi runtime giúp tiết kiệm 1.5GB VRAM.
> 4. **Hàm mất mát chuyên biệt:** Kết hợp $L_1 + \text{VGG Perceptual} + \text{EDT Loss}$ giúp hút nét vẽ về đúng vị trí và triệt tiêu bóng ma (ghosting).
> 5. **Hỗ trợ thời gian $t$ tùy ý theo Batch:** Vector hóa luồng chuyển động cho vector $t \in \mathbb{R}^{B \times 1 \times 1 \times 1}$ thay vì chạy vòng lặp tuần tự.

In [ ]:
# [CELL 1]: Cấu hình Môi trường, Thiết bị Phần cứng & Hạt giống Ngẫu nhiên
import os
import sys
import gc
import time
import math
import random
import shutil
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
import torchvision.models as models

# Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('=' * 65)
print(f'[*] PyTorch Version : {torch.__version__}')
print(f'[*] Execution Device: {device}')
if torch.cuda.is_available():
    print(f'[*] GPU Name        : {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'[*] VRAM Dung Lượng : {vram_gb:.2f} GB')
print('=' * 65)

In [ ]:
# [CELL 2]: Định vị và Kiểm tra Toàn vẹn Dữ liệu Dataset
KAGGLE_INPUT = '/kaggle/input'
WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './kaggle_output'
os.makedirs(WORKING_DIR, exist_ok=True)

# Tìm kiếm thư mục dataset tự động trong nhiều khả năng đường dẫn
candidate_paths = [
    os.path.join(KAGGLE_INPUT, 'anime-tps-finetune-dataset-ready'),
    os.path.join(KAGGLE_INPUT, 'anime_tps_finetune_dataset_ready'),
    os.path.join(KAGGLE_INPUT, 'pbl6-anime-tps-finetune-dataset-ready'),
    './data/processed/anime_tps_finetune_dataset_ready',
    '../data/processed/anime_tps_finetune_dataset_ready',
    '../../data/processed/anime_tps_finetune_dataset_ready'
]

DATASET_ROOT = None
for path in candidate_paths:
    if os.path.exists(path) and os.path.exists(os.path.join(path, 'triplets.csv')):
        DATASET_ROOT = path
        break

if DATASET_ROOT is None:
    # Duyệt tìm đệ quy nếu dataset nằm sâu trong thư mục con
    search_dir = KAGGLE_INPUT if os.path.exists(KAGGLE_INPUT) else '.'
    for root, dirs, files in os.walk(search_dir):
        if 'triplets.csv' in files:
            DATASET_ROOT = root
            break

if DATASET_ROOT is None:
    raise FileNotFoundError("❌ Không tìm thấy dataset chứa 'triplets.csv'! Vui lòng kiểm tra mục Input Datasets.")

print(f'[*] Đã kết nối Dataset tại: {DATASET_ROOT}')
df_meta = pd.read_csv(os.path.join(DATASET_ROOT, 'triplets.csv'))
print(f'[*] Tổng số Triplets: {len(df_meta)}')
print('[*] Phân bố Split:')
print(df_meta['split'].value_counts())
print(f'[*] Mẫu đầu tiên:\n{df_meta.iloc[0].to_dict()}')

In [ ]:
# [CELL 3]: Dataset & DataLoader Lazy Tối ưu Hóa Bộ nhớ Cực Hạn
class AnimeTPSDataset(Dataset):
    """
    Dataset đọc ảnh theo cơ chế Lazy Loading (On-the-fly):
    - Không bao giờ giữ mảng pixel trong RAM trước.
    - Đảo màu: Line drawing (nền trắng=255, nét=0) -> (nền=0.0, nét=1.0) cho TPS Model.
    - Train: Random Crop 512x512 để tiết kiệm VRAM và chống overfitting.
    - Val: Pad kích thước chia hết cho 32 (544x960) để đánh giá toàn khung hình.
    """
    def __init__(self, root_dir, split='train', crop_size=512):
        self.root_dir = root_dir
        self.split = split
        self.crop_size = crop_size
        df = pd.read_csv(os.path.join(root_dir, 'triplets.csv'))
        self.samples = df[df['split'] == split].reset_index(drop=True)

    def __len__(self):
        return len(self.samples)

    def _load_image(self, rel_path):
        # Đảm bảo đọc theo thư mục con tương ứng (train/ hoặc val/)
        full_path = os.path.join(self.root_dir, self.split, rel_path)
        if not os.path.exists(full_path):
            full_path = os.path.join(self.root_dir, rel_path)
        img = Image.open(full_path).convert('L')
        arr = np.array(img, dtype=np.float32) / 255.0
        # Đảo màu cho phù hợp đầu vào TPS (nền trắng -> 0, nét vẽ -> 1)
        arr = 1.0 - arr
        return arr

    def _load_matches(self, rel_path):
        full_path = os.path.join(self.root_dir, 'matches', rel_path)
        if not os.path.exists(full_path):
            full_path = os.path.join(self.root_dir, rel_path)
        kps = np.load(full_path).astype(np.float32)
        return kps # (2, N, 2) tọa độ [0, 1]

    def __getitem__(self, idx):
        row = self.samples.iloc[idx]
        t_val = float(row['t'])
        f0 = self._load_image(row['frame0'])
        fgt = self._load_image(row['frame_gt'])
        f1 = self._load_image(row['frame1'])
        kps = self._load_matches(row['match_file'])

        h, w = f0.shape

        if self.split == 'train':
            # Random Crop 512x512
            ch, cw = self.crop_size, self.crop_size
            if h > ch and w > cw:
                top = random.randint(0, h - ch)
                left = random.randint(0, w - cw)
                f0 = f0[top:top+ch, left:left+cw]
                fgt = fgt[top:top+ch, left:left+cw]
                f1 = f1[top:top+ch, left:left+cw]
                # Cập nhật tọa độ chuẩn hóa kps theo crop
                kps0 = kps[0].copy()
                kps1 = kps[1].copy()
                kps0[:, 0] = (kps0[:, 0] * w - left) / cw
                kps0[:, 1] = (kps0[:, 1] * h - top) / ch
                kps1[:, 0] = (kps1[:, 0] * w - left) / cw
                kps1[:, 1] = (kps1[:, 1] * h - top) / ch
                mask = (kps0[:, 0] >= 0) & (kps0[:, 0] <= 1) & (kps0[:, 1] >= 0) & (kps0[:, 1] <= 1) & \
                       (kps1[:, 0] >= 0) & (kps1[:, 0] <= 1) & (kps1[:, 1] >= 0) & (kps1[:, 1] <= 1)
                if np.sum(mask) >= 6:
                    kps = np.stack([kps0[mask], kps1[mask]], axis=0)
                else:
                    # Fallback nếu crop trúng vùng trống
                    kps = np.zeros((2, 0, 2), dtype=np.float32)
        else:
            # Validation: Pad to multiple of 32 (540->544, 960->960)
            pad_h = ((h + 31) // 32) * 32
            pad_w = ((w + 31) // 32) * 32
            if pad_h != h or pad_w != w:
                f0 = np.pad(f0, ((0, pad_h - h), (0, pad_w - w)), mode='constant')
                fgt = np.pad(fgt, ((0, pad_h - h), (0, pad_w - w)), mode='constant')
                f1 = np.pad(f1, ((0, pad_h - h), (0, pad_w - w)), mode='constant')
                # Cập nhật tỉ lệ chuẩn hóa
                kps0 = kps[0].copy()
                kps1 = kps[1].copy()
                kps0[:, 0] = (kps0[:, 0] * w) / pad_w
                kps0[:, 1] = (kps0[:, 1] * h) / pad_h
                kps1[:, 0] = (kps1[:, 0] * w) / pad_w
                kps1[:, 1] = (kps1[:, 1] * h) / pad_h
                kps = np.stack([kps0, kps1], axis=0)

        t0 = torch.from_numpy(f0).unsqueeze(0).float()   # (1, H, W)
        tgt = torch.from_numpy(fgt).unsqueeze(0).float() # (1, H, W)
        t1 = torch.from_numpy(f1).unsqueeze(0).float()   # (1, H, W)
        return t0, tgt, t1, torch.tensor(t_val, dtype=torch.float32), kps, row['clip_id']

def collate_fn_tps(batch):
    t0_list, tgt_list, t1_list, t_val_list, kps_list, clips = zip(*batch)
    t0 = torch.stack(t0_list, dim=0)
    tgt = torch.stack(tgt_list, dim=0)
    t1 = torch.stack(t1_list, dim=0)
    t_vals = torch.stack(t_val_list, dim=0).view(-1, 1, 1, 1)
    return t0, tgt, t1, t_vals, kps_list, clips

train_dataset = AnimeTPSDataset(DATASET_ROOT, split='train', crop_size=512)
val_dataset = AnimeTPSDataset(DATASET_ROOT, split='val', crop_size=512)

BATCH_SIZE = 4
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, collate_fn=collate_fn_tps, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=2, shuffle=False,
    num_workers=2, pin_memory=True, collate_fn=collate_fn_tps
)
print(f'[*] Số lượng Batches Huấn Luyện: {len(train_loader)} (Mỗi batch {BATCH_SIZE} mẫu)')
print(f'[*] Số lượng Batches Thẩm Định  : {len(val_loader)} (Mỗi batch 2 mẫu)')

In [ ]:
# [CELL 4]: Khối Giải Thuật Biến Dạng TPS Solver & Warp Layer (utils.py)
def grid_from_flow(flow, size):
    b, h, w, _ = flow.shape
    grid_y, grid_x = torch.meshgrid(
        torch.linspace(-1, 1, h, device=flow.device),
        torch.linspace(-1, 1, w, device=flow.device), indexing='ij'
    )
    grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0).repeat(b, 1, 1, 1)
    grid = grid + flow
    return grid

def tps_rbf(r):
    r = torch.clamp(r, min=1e-6)
    return (r ** 2) * torch.log(r)

def tps_theta_from_points(c_src, c_dst, lambd=0.2):
    """
    Giải hệ phương trình Thin-Plate Spline tìm ma trận trọng số biến dạng.
    Áp dụng điều chuẩn lambd=0.2 để chống rung lắc méo nét.
    """
    device = c_src.device
    n = c_src.shape[0]
    if n < 4:
        return torch.zeros((n + 3, 2), device=device)
    diff = c_src.unsqueeze(1) - c_src.unsqueeze(0)
    dist = torch.norm(diff, dim=-1)
    K = tps_rbf(dist)
    K = K + lambd * torch.eye(n, device=device)
    P = torch.cat([torch.ones((n, 1), device=device), c_src], dim=-1)
    L_top = torch.cat([K, P], dim=-1)
    L_bot = torch.cat([P.t(), torch.zeros((3, 3), device=device)], dim=-1)
    L = torch.cat([L_top, L_bot], dim=0)
    Y = torch.cat([c_dst, torch.zeros((3, 2), device=device)], dim=0)
    try:
        theta = torch.linalg.solve(L, Y)
    except RuntimeError:
        theta = torch.zeros((n + 3, 2), device=device)
    return theta

def tps_flow_from_theta(theta, c_src, h, w, device, chunk_size=16384):
    """
    Sinh trường dòng chuyển động TPS coarse flow (1, 2, H, W) trong torch.no_grad().
    Áp dụng Pixel Chunking (16384 px/lần) để triệt tiêu hoàn toàn CUDA OOM trên ảnh 544x960.
    """
    n = c_src.shape[0]
    if n < 4 or theta.shape[0] != (n + 3):
        return torch.zeros((1, 2, h, w), device=device)
    grid_y, grid_x = torch.meshgrid(
        torch.linspace(0, 1, h, device=device),
        torch.linspace(0, 1, w, device=device), indexing='ij'
    )
    pts = torch.stack([grid_x, grid_y], dim=-1).view(-1, 2) # (HW, 2)
    total_pts = pts.shape[0]
    
    flow_chunks = []
    for i in range(0, total_pts, chunk_size):
        p_chunk = pts[i:i + chunk_size]                      # (C, 2)
        diff = p_chunk.unsqueeze(1) - c_src.unsqueeze(0)     # (C, n, 2)
        dist = torch.norm(diff, dim=-1)                      # (C, n)
        U = tps_rbf(dist)                                    # (C, n)
        P = torch.cat([torch.ones((p_chunk.shape[0], 1), device=device), p_chunk], dim=-1) # (C, 3)
        basis = torch.cat([U, P], dim=-1)                    # (C, n+3)
        mapped = torch.matmul(basis, theta)                  # (C, 2)
        flow_chunk = mapped - p_chunk                        # (C, 2)
        flow_chunks.append(flow_chunk)
        
    flow = torch.cat(flow_chunks, dim=0)                     # (HW, 2)
    flow = flow.view(h, w, 2).permute(2, 0, 1).unsqueeze(0) # (1, 2, H, W)
    return flow

def backward_warp(img, flow):
    b, _, h, w = img.shape
    flow_perm = flow.permute(0, 2, 3, 1) # (B, H, W, 2)
    grid = grid_from_flow(flow_perm, (b, 1, h, w))
    warped = F.grid_sample(img, grid, padding_mode='border', align_corners=True)
    return warped

print('[*] TPS Solver & Warp Layer đã sẵn sàng (Đã tối ưu Chunking chống OOM).')


In [ ]:
# [CELL 5]: Kiến trúc Mô hình TPS-Inbetween (Contextnet, IFNet, Unet)
def conv(in_p, out_p, kernel_size=3, stride=1, padding=1, dilation=1):
    return nn.Sequential(
        nn.Conv2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation, bias=True),
        nn.PReLU(out_p)
    )

def deconv(in_p, out_p, kernel_size=4, stride=2, padding=1):
    return nn.Sequential(
        nn.ConvTranspose2d(in_p, out_p, kernel_size=kernel_size, stride=stride, padding=padding, bias=True),
        nn.PReLU(out_p)
    )

class Conv2(nn.Module):
    def __init__(self, in_p, out_p, stride=2):
        super(Conv2, self).__init__()
        self.conv1 = conv(in_p, out_p, 3, stride, 1)
        self.conv2 = conv(out_p, out_p, 3, 1, 1)
    def forward(self, x):
        return self.conv2(self.conv1(x))

class Contextnet(nn.Module):
    def __init__(self, indim=1, c=16):
        super(Contextnet, self).__init__()
        self.conv1 = Conv2(indim, c)
        self.conv2 = Conv2(c, 2*c)
        self.conv3 = Conv2(2*c, 4*c)
        self.conv4 = Conv2(4*c, 8*c)
    def forward(self, x, flow):
        x = self.conv1(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f1 = backward_warp(x, flow)
        x = self.conv2(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f2 = backward_warp(x, flow)
        x = self.conv3(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f3 = backward_warp(x, flow)
        x = self.conv4(x)
        flow = F.interpolate(flow, scale_factor=0.5, mode='bilinear', align_corners=False) * 0.5
        f4 = backward_warp(x, flow)
        return [f1, f2, f3, f4]

class IFBlock(nn.Module):
    def __init__(self, in_planes, c=64):
        super(IFBlock, self).__init__()
        self.conv0 = nn.Sequential(
            conv(in_planes, c//2, 3, 2, 1),
            conv(c//2, c, 3, 2, 1),
        )
        self.convblock = nn.Sequential(*[conv(c, c) for _ in range(8)])
        self.lastconv = nn.ConvTranspose2d(c, 5, 4, 2, 1)

    def forward(self, x, flow, scale):
        if scale != 1:
            x = F.interpolate(x, scale_factor = 1. / scale, mode='bilinear', align_corners=False)
        if flow is not None:
            flow = F.interpolate(flow, scale_factor = 1. / scale, mode='bilinear', align_corners=False) * (1. / scale)
            x = torch.cat((x, flow), 1)
        x = self.conv0(x)
        x = self.convblock(x) + x
        tmp = self.lastconv(x)
        tmp = F.interpolate(tmp, scale_factor = scale * 2, mode='bilinear', align_corners=False)
        flow = tmp[:, :4] * scale * 2
        mask = tmp[:, 4:5]
        return flow, mask

class IFNet(nn.Module):
    def __init__(self):
        super(IFNet, self).__init__()
        self.block0 = IFBlock(6, c=240)
        self.block1 = IFBlock(13+4, c=150)
        self.block2 = IFBlock(13+4, c=90)

    def forward(self, I0t, I1t, scale=[4, 2, 1]):
        x = torch.cat((I0t.repeat(1, 3, 1, 1), I1t.repeat(1, 3, 1, 1)), dim=1)
        img0 = x[:, :3]
        img1 = x[:, 3:6]
        flow_list = []
        mask_list = []
        warped_img0 = img0
        warped_img1 = img1
        flow = None
        stu = [self.block0, self.block1, self.block2]
        for i in range(3):
            if flow is not None:
                flow_d, mask_d = stu[i](torch.cat((img0, img1, warped_img0, warped_img1, mask), 1), flow, scale=scale[i])
                flow = flow + flow_d
                mask = mask + mask_d
            else:
                flow, mask = stu[i](torch.cat((img0, img1), 1), None, scale=scale[i])
            mask_list.append(torch.sigmoid(mask))
            flow_list.append(flow)
            warped_img0 = backward_warp(img0, flow[:, :2])
            warped_img1 = backward_warp(img1, flow[:, 2:4])
        return flow_list[2], mask_list[2]

class SynthesisUnet(nn.Module):
    def __init__(self, indim=9, outdim=1, c=16):
        super(SynthesisUnet, self).__init__()
        self.down0 = Conv2(indim, 2*c)
        self.down1 = Conv2(4*c, 4*c)
        self.down2 = Conv2(8*c, 8*c)
        self.down3 = Conv2(16*c, 16*c)
        self.up0 = deconv(32*c, 8*c)
        self.up1 = deconv(16*c, 4*c)
        self.up2 = deconv(8*c, 2*c)
        self.up3 = deconv(4*c, c)
        self.conv = nn.Conv2d(c, outdim, 3, 1, 1)
    def forward(self, img0, img1, flow0, flow1, warped0, warped1, mask, c0, c1):
        s0 = self.down0(torch.cat((img0, img1, warped0, warped1, flow0, flow1, mask), 1))
        s1 = self.down1(torch.cat((s0, c0[0], c1[0]), 1))
        s2 = self.down2(torch.cat((s1, c0[1], c1[1]), 1))
        s3 = self.down3(torch.cat((s2, c0[2], c1[2]), 1))
        x = self.up0(torch.cat((s3, c0[3], c1[3]), 1))
        x = self.up1(torch.cat((x, s2), 1))
        x = self.up2(torch.cat((x, s1), 1))
        x = self.up3(torch.cat((x, s0), 1))
        return torch.sigmoid(self.conv(x))

class TPSInbetweenFinetune(nn.Module):
    def __init__(self):
        super(TPSInbetweenFinetune, self).__init__()
        self.encoder = Contextnet(indim=1)
        self.IFNet = IFNet()
        self.unet = SynthesisUnet(indim=9, outdim=1)

    def forward(self, x0, x1, t_vals, kps_list):
        b, c, h, w = x0.shape
        dev = x0.device

        # 1. Tính toán TPS Coarse Motion trong no_grad (Non-parametric)
        flow01_list, flow10_list = [], []
        with torch.no_grad():
            for i in range(b):
                kps = kps_list[i]
                if kps is None or len(kps) < 2 or kps.shape[1] < 4:
                    flow01_ = torch.zeros((1, 2, h, w), device=dev)
                    flow10_ = torch.zeros((1, 2, h, w), device=dev)
                else:
                    c0 = torch.from_numpy(kps[0]).to(dev)
                    c1 = torch.from_numpy(kps[1]).to(dev)
                    # Giới hạn tối đa 150 keypoints quan trọng nhất để chống OOM và giảm nhiễu
                    if c0.shape[0] > 150:
                        idx_sub = torch.linspace(0, c0.shape[0] - 1, 150, dtype=torch.long, device=dev)
                        c0 = c0[idx_sub]
                        c1 = c1[idx_sub]
                    theta01 = tps_theta_from_points(c0, c1, lambd=0.2)
                    theta10 = tps_theta_from_points(c1, c0, lambd=0.2)
                    flow01_ = tps_flow_from_theta(theta01, c0, h, w, dev)
                    flow10_ = tps_flow_from_theta(theta10, c1, h, w, dev)
                flow01_list.append(flow01_)
                flow10_list.append(flow10_)

        coarse_flow01 = torch.cat(flow01_list, dim=0)
        coarse_flow10 = torch.cat(flow10_list, dim=0)

        # 2. Vector hóa nội suy thời gian t tùy ý (Hỗ trợ toàn bộ Batch cùng lúc)
        coarse_flow0t = coarse_flow01 * (t_vals ** 2) - (1.0 - t_vals) * t_vals * coarse_flow10
        coarse_flow1t = coarse_flow10 * ((1.0 - t_vals) ** 2) - t_vals * (1.0 - t_vals) * coarse_flow01

        coarse_I0t = backward_warp(x0, coarse_flow0t)
        coarse_I1t = backward_warp(x1, coarse_flow1t)

        # 3. Fine Motion Refinement qua IFNet
        feats0 = self.encoder(x0, coarse_flow0t)
        feats1 = self.encoder(x1, coarse_flow1t)
        flow_refine, mask = self.IFNet(coarse_I0t, coarse_I1t)

        fine_I0t = backward_warp(coarse_I0t, flow_refine[:, :2])
        fine_I1t = backward_warp(coarse_I1t, flow_refine[:, 2:4])

        # 4. Tổng hợp & Alpha blending qua Synthesis Unet
        refine_out = self.unet(
            x0, x1,
            coarse_flow0t + flow_refine[:, :2],
            coarse_flow1t + flow_refine[:, 2:4],
            fine_I0t, fine_I1t, mask, feats0, feats1
        )
        res = refine_out[:, 0:1, :, :] * 2.0 - 1.0
        pred_t = (1.0 - t_vals) * fine_I0t + t_vals * fine_I1t + res
        pred_t = torch.clamp(pred_t, 0.0, 1.0)
        return pred_t

model = TPSInbetweenFinetune().to(device)
print('[*] Mô hình TPSInbetweenFinetune đã được khởi tạo thành công.')


In [ ]:
# [CELL 6]: Các Hàm Mất Mát Chuyên Biệt cho Line Art (L1 + VGG Perceptual + EDT Loss)
class VGGPerceptualLoss(nn.Module):
    """
    VGG-16 Perceptual Loss đóng băng hoàn toàn để đo lường cấu trúc thị giác.
    """
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        vgg = models.vgg16(pretrained=True).features
        self.slice1 = nn.Sequential()
        self.slice2 = nn.Sequential()
        for x in range(4):
            self.slice1.add_module(str(x), vgg[x])
        for x in range(4, 9):
            self.slice2.add_module(str(x), vgg[x])
        for param in self.parameters():
            param.requires_grad = False
        self.eval()

    def forward(self, pred, target):
        # Chuyển grayscale 1 channel thành 3 channels cho VGG
        pred_3c = pred.repeat(1, 3, 1, 1)
        target_3c = target.repeat(1, 3, 1, 1)
        h_pred = self.slice1(pred_3c)
        h_target = self.slice1(target_3c)
        loss = F.l1_loss(h_pred, h_target)
        h_pred = self.slice2(h_pred)
        h_target = self.slice2(h_target)
        loss += F.l1_loss(h_pred, h_target)
        return loss

def compute_batch_edt_loss(pred, target):
    """
    Euclidean Distance Transform (EDT) Loss:
    Hút các nét vẽ bị lệch của pred về đúng vị trí ground truth target.
    Tính toán trên CPU với scipy rồi chuyển sang GPU Tensor detached.
    """
    b = target.shape[0]
    edt_maps = []
    with torch.no_grad():
        target_np = target.detach().cpu().numpy() # (B, 1, H, W)
        for i in range(b):
            # Vùng nét vẽ là nơi target > 0.1
            line_mask = (target_np[i, 0] > 0.1).astype(np.uint8)
            if np.sum(line_mask) == 0:
                edt = np.zeros_like(line_mask, dtype=np.float32)
            else:
                # Khoảng cách từ mỗi pixel tới nét vẽ gần nhất
                edt = distance_transform_edt(1 - line_mask).astype(np.float32)
                # Chuẩn hóa khoảng cách (clamping để tránh gradient quá lớn)
                edt = np.clip(edt / 20.0, 0.0, 1.0)
            edt_maps.append(edt)
        edt_tensor = torch.from_numpy(np.stack(edt_maps, axis=0)).unsqueeze(1).to(pred.device)
    # Loss tỉ lệ thuận với cường độ nét dự đoán nhân với khoảng cách tới GT
    edt_loss = torch.mean(pred * (edt_tensor ** 2))
    return edt_loss

vgg_loss_fn = VGGPerceptualLoss().to(device)
print('[*] Các hàm Loss (L1, VGG, EDT) đã được khởi tạo.')

In [ ]:
# [CELL 7]: Thiết lập Optimizer, AMP Scaler, Chiến lược Đóng băng & Nạp Trọng số Gốc
# 1. Đóng băng encoder Contextnet
for param in model.encoder.parameters():
    param.requires_grad = False
print('[*] Đã đóng băng khối Contextnet Encoder.')

# 2. Nạp trọng số gốc (nếu có file checkpoint sẵn)
pretrained_paths = [
    os.path.join(KAGGLE_INPUT, 'tps-baseline-model', 'model_latest.pt'),
    os.path.join(KAGGLE_INPUT, 'tps-inbetween-weights', 'model_latest.pt'),
    './checkpoints/pretrained_baselines/tps-original/model_latest.pt'
]
loaded_ckpt = False
for ckpt_p in pretrained_paths:
    if os.path.exists(ckpt_p):
        try:
            state = torch.load(ckpt_p, map_location=device)
            model.load_state_dict(state, strict=False)
            print(f'[*] Đã nạp thành công trọng số khởi tạo từ: {ckpt_p}')
            loaded_ckpt = True
            break
        except Exception as e:
            print(f'[!] Thử nạp {ckpt_p} gặp lỗi: {e}')
if not loaded_ckpt:
    print('[*] Huấn luyện từ trạng thái trọng số ngẫu nhiên/hiện tại.')

# 2.1. Thiết lập thư mục lưu trữ Checkpoint chuyên biệt
CHECKPOINT_DIR = os.path.join(WORKING_DIR, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'[*] Thư mục lưu trữ Checkpoints: {CHECKPOINT_DIR}')

# 3. Optimizer với 2 nhóm Learning Rate riêng biệt
optimizer = AdamW([
    {'params': model.IFNet.parameters(), 'lr': 2e-5, 'weight_decay': 1e-4},
    {'params': model.unet.parameters(),  'lr': 5e-5, 'weight_decay': 1e-4}
])

TOTAL_EPOCHS = 25
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
scaler = GradScaler() # Hỗ trợ FP16 Mixed Precision


In [ ]:
# [CELL 8]: Hàm Tính Chỉ Số Đo Lường (PSNR, SSIM, Ghost Pixels)
def calculate_metrics(pred_tensor, gt_tensor):
    """
    Tính PSNR, SSIM và đếm số lượng Ghost Pixels trên batch ảnh.
    Lưu ý: pred_tensor và gt_tensor đang ở dạng (nền=0, nét=1).
    Ta đảo ngược lại (1.0 - x) * 255.0 để đo lường trên không gian ảnh vẽ thực tế (nền trắng=255).
    """
    with torch.no_grad():
        # Đảo màu về ảnh nét vẽ truyền thống
        p_img = ((1.0 - pred_tensor) * 255.0).clamp(0, 255).byte().cpu().numpy()
        g_img = ((1.0 - gt_tensor) * 255.0).clamp(0, 255).byte().cpu().numpy()

        b = p_img.shape[0]
        psnr_list, ghost_list = [], []
        for i in range(b):
            p = p_img[i, 0].astype(np.float32)
            g = g_img[i, 0].astype(np.float32)
            mse = np.mean((p - g) ** 2)
            if mse == 0:
                psnr = 50.0
            else:
                psnr = 20.0 * np.log10(255.0 / np.sqrt(mse))
            psnr_list.append(psnr)

            # Ghosting: đếm pixel xám mờ trên nền trắng (khoảng 180 ~ 250)
            ghost_mask = (p >= 180) & (p <= 250) & (g > 250)
            ghost_list.append(np.sum(ghost_mask))

        return float(np.mean(psnr_list)), float(np.mean(ghost_list))

print('[*] Hàm tính toán kiểm toán chất lượng đã sẵn sàng.')

In [ ]:
# [CELL 9]: Quản Lý Checkpoint Tự Động Theo Cột Mốc Tiến Bộ (Smart Checkpoint Manager)
class SmartCheckpointManager:
    """
    Trình quản lý lưu Checkpoint thông minh cho Fine-tuning TPS-Inbetween.
    Tiêu chuẩn ghi nhận tiến bộ tốt:
      1. Kỷ lục PSNR cao nhất toàn diện (model_tps_v0.1_best.pt)
      2. Kỷ lục nét sạch nhất / ít ghosting nhất (model_tps_v0.1_best_cleanlines.pt)
      3. Bước nhảy vọt PSNR >= +0.5 dB so với mốc gần nhất (Milestones)
      4. Vượt qua các mốc chất lượng tuyệt đối: 15.0 dB (khởi sắc), 17.5 dB (ổn định), 19.0 dB (xuất sắc)
      5. Giới hạn giữ Top-K checkpoints (mặc định Top 3) để tránh tràn ổ đĩa Kaggle (19.5GB limit).
    """
    def __init__(self, save_dir, keep_top_k=3, min_delta_psnr=0.5, absolute_tiers=[15.0, 17.5, 19.0]):
        self.save_dir = save_dir
        os.makedirs(save_dir, exist_ok=True)
        self.keep_top_k = keep_top_k
        self.min_delta_psnr = min_delta_psnr
        self.absolute_tiers = sorted(absolute_tiers)
        self.reached_tiers = set()
        
        self.best_psnr = -1.0
        self.last_milestone_psnr = -1.0
        self.best_ghost = float('inf')
        self.saved_milestones = [] # [(psnr, file_path)]

    def step(self, epoch, model, optimizer, psnr, ghost):
        saved_reasons = []
        
        # 1. Kỷ lục PSNR cao nhất toàn diện
        if psnr > self.best_psnr:
            delta = psnr - self.best_psnr if self.best_psnr > 0 else 0
            self.best_psnr = psnr
            best_path = os.path.join(self.save_dir, 'model_tps_v0.1_best.pt')
            torch.save(model.state_dict(), best_path)
            # Lưu thêm 1 bản copy tại WORKING_DIR để các cell đánh giá truy cập thuận tiện
            working_best = os.path.join(WORKING_DIR, 'model_tps_v0.1_best.pt')
            if working_best != best_path:
                shutil.copyfile(best_path, working_best)
            saved_reasons.append(f'🌟 [KỶ LỤC PSNR TOÀN DIỆN]: {psnr:.2f} dB (+{delta:.2f} dB)')
            
        # 2. Kỷ lục Nét sạch nhất (Ghost Pixels thấp nhất)
        if ghost < self.best_ghost:
            self.best_ghost = ghost
            clean_path = os.path.join(self.save_dir, 'model_tps_v0.1_best_cleanlines.pt')
            torch.save(model.state_dict(), clean_path)
            saved_reasons.append(f'🎯 [KỶ LỤC SẠCH NÉT]: Ghost Pixels = {ghost:.0f}')

        # 3. Chạm các Cột mốc Chất lượng Tuyệt đối (15dB, 17.5dB, 19dB)
        for tier in self.absolute_tiers:
            if tier not in self.reached_tiers and psnr >= tier:
                self.reached_tiers.add(tier)
                tier_filename = f'model_tps_v0.1_tier_{tier:.1f}dB_ep{epoch:02d}.pt'
                tier_path = os.path.join(self.save_dir, tier_filename)
                torch.save(model.state_dict(), tier_path)
                saved_reasons.append(f'🏅 [CỘT MỐC CHẤT LƯỢNG {tier:.1f} dB]: Đã vượt mốc tại Epoch {epoch:02d}')

        # 4. Tiến bộ vượt bậc tương đối (Delta >= min_delta_psnr)
        if self.last_milestone_psnr < 0 or (psnr - self.last_milestone_psnr) >= self.min_delta_psnr:
            self.last_milestone_psnr = psnr
            milestone_filename = f'model_tps_v0.1_ep{epoch:02d}_psnr{psnr:.2f}.pt'
            milestone_path = os.path.join(self.save_dir, milestone_filename)
            torch.save(model.state_dict(), milestone_path)
            self.saved_milestones.append((psnr, milestone_path))
            saved_reasons.append(f'🚀 [TIẾN BỘ VƯỢT BẬC]: Đạt {psnr:.2f} dB (>= +{self.min_delta_psnr:.1f} dB)')
            
            # Quản lý Top-K: Xóa checkpoint milestone có PSNR thấp nhất nếu vượt quá keep_top_k
            if len(self.saved_milestones) > self.keep_top_k:
                self.saved_milestones.sort(key=lambda x: x[0])
                worst_psnr, worst_path = self.saved_milestones.pop(0)
                if os.path.exists(worst_path) and not worst_path.endswith('_best.pt'):
                    try:
                        os.remove(worst_path)
                    except Exception:
                        pass

        # 5. Luôn lưu Latest Checkpoint
        latest_path = os.path.join(self.save_dir, 'model_tps_v0.1_latest.pt')
        torch.save(model.state_dict(), latest_path)
        working_latest = os.path.join(WORKING_DIR, 'model_tps_v0.1_latest.pt')
        if working_latest != latest_path:
            shutil.copyfile(latest_path, working_latest)

        # In thông báo các thành tựu đạt được
        if saved_reasons:
            for reason in saved_reasons:
                print(f'    💾 {reason}')

checkpoint_manager = SmartCheckpointManager(
    save_dir=CHECKPOINT_DIR,
    keep_top_k=3,
    min_delta_psnr=0.5,
    absolute_tiers=[15.0, 17.5, 19.0]
)
print(f'[*] Đã kích hoạt SmartCheckpointManager tại: {CHECKPOINT_DIR}')
print(f'    - Mức tăng tối thiểu lưu mốc mới: >= +0.5 dB')
print(f'    - Các mốc chất lượng tuyệt đối     : 15.0 dB (Khởi sắc), 17.5 dB (Ổn định), 19.0 dB (Xuất sắc)')
print(f'    - Giới hạn lưu trữ Top-K            : Tối đa 3 Milestone Checkpoints (Chống đầy đĩa)')


In [ ]:
# [CELL 10]: Vòng Lặp Huấn Luyện & Thẩm Định Hoàn Chỉnh (25 Epochs) với ETA & TQDM
import datetime
from tqdm.auto import tqdm

GRAD_ACCUM_STEPS = 2
history = []
train_start_time = time.time()

print('=' * 75)
print(f'🚀 BẮT ĐẦU HUẤN LUYỆN TINH CHỈNH TPS-INBETWEEN v0.1 ({TOTAL_EPOCHS} EPOCHS)')
print(f'   - Effective Batch Size: {BATCH_SIZE * GRAD_ACCUM_STEPS}')
print('   - Mixed Precision     : PyTorch AMP FP16 Enabled (torch.amp.autocast)')
print('   - Cơ chế bảo vệ VRAM  : Pixel Chunking + Auto Cache Cleanup')
print('   - Giám sát tiến độ    : Tích hợp tqdm Progress Bar & Ước lượng ETA')
print('=' * 75)

for epoch in range(1, TOTAL_EPOCHS + 1):
    epoch_start_time = time.time()
    model.train()
    train_loss_accum = 0.0
    train_steps = 0
    optimizer.zero_grad()

    # Progress Bar cho tập huấn luyện (có ETA thời gian còn lại từng giây)
    train_pbar = tqdm(
        train_loader,
        desc=f"🏋️ [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Train",
        unit="batch",
        leave=False
    )

    for step, (t0, tgt, t1, t_vals, kps_list, clips) in enumerate(train_pbar):
        t0 = t0.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        t1 = t1.to(device, non_blocking=True)
        t_vals = t_vals.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            pred = model(t0, t1, t_vals, kps_list)
            loss_l1 = F.l1_loss(pred, tgt)
            loss_vgg = vgg_loss_fn(pred, tgt)
            loss_edt = compute_batch_edt_loss(pred, tgt)
            total_loss = loss_l1 + 0.1 * loss_vgg + 0.4 * loss_edt
            loss_scaled = total_loss / GRAD_ACCUM_STEPS

        scaler.scale(loss_scaled).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        train_loss_accum += total_loss.item()
        train_steps += 1

        train_pbar.set_postfix({
            'loss': f"{total_loss.item():.4f}",
            'l1': f"{loss_l1.item():.4f}",
            'edt': f"{loss_edt.item():.4f}"
        })

    train_loss_avg = train_loss_accum / max(1, train_steps)
    scheduler.step()

    # --- GIẢI PHÓNG VRAM TỪ TRAIN TRƯỚC KHI VÀO VAL ---
    torch.cuda.empty_cache()
    gc.collect()

    # --- GIAI ĐOẠN VALIDATION TRÊN TẬP VAL (508 TRIPLETS) ---
    model.eval()
    val_psnr_accum = 0.0
    val_ghost_accum = 0.0
    val_steps = 0

    val_pbar = tqdm(
        val_loader,
        desc=f"🔍 [Ep {epoch:02d}/{TOTAL_EPOCHS:02d}] Val  ",
        unit="batch",
        leave=False
    )

    with torch.no_grad():
        for t0, tgt, t1, t_vals, kps_list, clips in val_pbar:
            t0 = t0.to(device, non_blocking=True)
            tgt = tgt.to(device, non_blocking=True)
            t1 = t1.to(device, non_blocking=True)
            t_vals = t_vals.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                pred = model(t0, t1, t_vals, kps_list)
            psnr, ghost = calculate_metrics(pred, tgt)
            val_psnr_accum += psnr
            val_ghost_accum += ghost
            val_steps += 1
            val_pbar.set_postfix({'psnr': f"{psnr:.2f}dB"})

    val_psnr_avg = val_psnr_accum / max(1, val_steps)
    val_ghost_avg = val_ghost_accum / max(1, val_steps)
    
    # Tính ETA thời gian còn lại cho toàn bộ 25 Epochs
    epoch_elapsed = time.time() - epoch_start_time
    total_elapsed = time.time() - train_start_time
    avg_epoch_sec = total_elapsed / epoch
    rem_epochs = TOTAL_EPOCHS - epoch
    eta_total_sec = avg_epoch_sec * rem_epochs
    eta_str = str(datetime.timedelta(seconds=int(eta_total_sec)))

    print(f'[*] [Epoch {epoch:02d}/{TOTAL_EPOCHS:02d}] ({epoch_elapsed:.1f}s | ETA còn lại: {eta_str}) ' \
          f'| Loss: {train_loss_avg:.4f} | Val PSNR: {val_psnr_avg:.2f} dB | Ghost Px: {val_ghost_avg:.0f}')

    # Ghi nhận lịch sử huấn luyện
    history.append({
        'epoch': epoch,
        'train_loss': train_loss_avg,
        'val_psnr': val_psnr_avg,
        'val_ghost_pixels': val_ghost_avg,
        'lr': optimizer.param_groups[1]['lr'],
        'time_sec': epoch_elapsed
    })

    # Tự động thẩm định và lưu Checkpoint theo Cột mốc tiến bộ
    checkpoint_manager.step(
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        psnr=val_psnr_avg,
        ghost=val_ghost_avg
    )

    # Dọn dẹp VRAM sau mỗi Epoch
    torch.cuda.empty_cache()
    gc.collect()

print('=' * 75)
print(f'🎉 HUẤN LUYỆN HOÀN TẤT! Điểm PSNR Thẩm định tốt nhất đạt: {checkpoint_manager.best_psnr:.2f} dB')
print(f'   Tổng thời gian thực hiện: {str(datetime.timedelta(seconds=int(time.time() - train_start_time)))}')
print('=' * 75)


In [ ]:
# [CELL 11]: Kiểm Toán Trực Quan Định Tính & So Sánh Song Song
visual_dir = os.path.join(WORKING_DIR, 'visual_samples')
os.makedirs(visual_dir, exist_ok=True)

# Nạp Best Model để kiểm tra suy luận
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'model_tps_v0.1_best.pt')
if not os.path.exists(best_ckpt_path):
    best_ckpt_path = os.path.join(WORKING_DIR, 'model_tps_v0.1_best.pt')
if os.path.exists(best_ckpt_path):
    model.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    print(f'[*] Đã nạp Best Checkpoint để kiểm toán: {best_ckpt_path}')
model.eval()

# Chọn 3 mẫu validation ngẫu nhiên để trực quan hóa
sample_indices = [0, min(10, len(val_dataset)-1), min(25, len(val_dataset)-1)]

plt.figure(figsize=(18, 6 * len(sample_indices)))
plot_row = 1

with torch.no_grad():
    for idx in sample_indices:
        t0, tgt, t1, t_val, kps, clip_id = val_dataset[idx]
        t0_b = t0.unsqueeze(0).to(device)
        t1_b = t1.unsqueeze(0).to(device)
        t_val_b = t_val.view(1, 1, 1, 1).to(device)

        with autocast():
            pred_b = model(t0_b, t1_b, t_val_b, [kps])

        # Đảo ngược màu về ảnh trắng nền đen nét
        img_0 = (1.0 - t0[0].numpy()) * 255.0
        img_pred = ((1.0 - pred_b[0, 0].cpu().float().numpy()) * 255.0).clip(0, 255)
        img_gt = (1.0 - tgt[0].numpy()) * 255.0
        diff = np.abs(img_pred - img_gt)

        plt.subplot(len(sample_indices), 4, (plot_row - 1) * 4 + 1)
        plt.imshow(img_0, cmap='gray')
        plt.title(f'Keyframe 0 (t=0.0) | {clip_id}')
        plt.axis('off')

        plt.subplot(len(sample_indices), 4, (plot_row - 1) * 4 + 2)
        plt.imshow(img_pred, cmap='gray')
        plt.title(f'TPS Fine-tuned (t={t_val.item():.2f})')
        plt.axis('off')

        plt.subplot(len(sample_indices), 4, (plot_row - 1) * 4 + 3)
        plt.imshow(img_gt, cmap='gray')
        plt.title(f'Ground Truth (t={t_val.item():.2f})')
        plt.axis('off')

        plt.subplot(len(sample_indices), 4, (plot_row - 1) * 4 + 4)
        plt.imshow(diff, cmap='hot')
        plt.title(f'Absolute Difference Map')
        plt.axis('off')

        plot_row += 1

preview_save_path = os.path.join(visual_dir, 'val_comparison_preview.png')
plt.tight_layout()
plt.savefig(preview_save_path, dpi=150)
plt.show()


In [ ]:
# [CELL 12]: Đóng Gói Thành Phẩm & Xuất File Tải Về Trực Tiếp
# 1. Ghi nhận file lịch sử training
df_history = pd.DataFrame(history)
history_csv = os.path.join(WORKING_DIR, 'training_history.csv')
df_history.to_csv(history_csv, index=False)
print(f'[*] Đã xuất lịch sử huấn luyện vào: {history_csv}')

# 2. Đóng gói kết quả thành file ZIP sẵn sàng download
zip_path = os.path.join(WORKING_DIR, 'tps_inbetween_finetune_v0.1_results.zip')
print('[*] Đang đóng gói tệp kết quả và tất cả checkpoints tiến bộ...')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Thêm toàn bộ các Checkpoint trong thư mục checkpoints/
    if os.path.exists(CHECKPOINT_DIR):
        for root, dirs, files in os.walk(CHECKPOINT_DIR):
            for file in files:
                f_path = os.path.join(root, file)
                zipf.write(f_path, arcname=os.path.join('checkpoints', file))
                print(f'    + Đã gói checkpoint: {file}')
    if os.path.exists(history_csv):
        zipf.write(history_csv, arcname='training_history.csv')
    if os.path.exists(preview_save_path):
        zipf.write(preview_save_path, arcname='val_comparison_preview.png')

zip_size_mb = os.path.getsize(zip_path) / (1024**2)
print('=' * 75)
print(f'📦 GÓI KẾT QUẢ ĐÃ SẴN SÀNG: {zip_path} ({zip_size_mb:.2f} MB)')
print('   - Chứa toàn bộ Checkpoints tiến bộ : checkpoints/*.pt')
print('   - Chứa Lịch sử Huấn Luyện          : training_history.csv')
print('   - Chứa Ảnh kiểm toán so sánh       : val_comparison_preview.png')
print('👉 Bạn có thể tải file ZIP này trực tiếp từ tab "Output" của Kaggle!')
print('=' * 75)
